In [ ]:
!pip install anthropic
!pip install google-generativeai

In [46]:
import asyncio
import json
import logging
import os
import pandas as pd
from typing import List, Dict, Any, Optional
from openai import AsyncOpenAI
import google.generativeai as genai
from abc import ABC, abstractmethod
from anthropic import AsyncAnthropic

# Cấu hình Logging
logging.basicConfig(level=logging.INFO)

# ==========================================
# 1. MULTI-PROVIDER INTERFACE
# ==========================================
class LLMProvider(ABC):
    @abstractmethod
    async def generate_response(self, system_prompt: str, user_prompt: str) -> Dict:
        pass

    @abstractmethod
    async def health_check(self) -> bool:
        pass

    def _extract_json(self, text: str) -> Any:
        """Hỗ trợ bóc tách cả JSON Object {} và JSON Array []"""
        try:
            # Tìm vị trí mở đầu của Object hoặc Array
            start_brace = text.find('{')
            start_bracket = text.find('[')

            # Xác định cái nào xuất hiện trước
            if start_brace == -1 and start_bracket == -1:
                raise ValueError("Không tìm thấy JSON.")

            if start_brace != -1 and (start_bracket == -1 or start_brace < start_bracket):
                start = start_brace
                end = text.rfind('}') + 1
            else:
                start = start_bracket
                end = text.rfind(']') + 1

            return json.loads(text[start:end])
        except Exception as e:
            logging.error(f"Lỗi phân tách JSON: {e} | Nội dung: {text[:200]}...")
            return None

# --- OpenAI Provider ---
class OpenAIProvider(LLMProvider):
    def __init__(self, api_key: str, model: str = "gpt-4o-mini"):
        self.client = AsyncOpenAI(api_key=api_key)
        self.model = model

    async def health_check(self):
        print(f"\n--- Checking OpenAI ({self.model}) ---")
        try:
            response = await self.client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": "ping"}],
                max_tokens=10
            )
            print("✅ OpenAI API OK")
            print(f"Response: {response.choices[0].message.content}")
            return True
        except Exception as e:
            print(f"❌ OpenAI Error: {e}")
            return False

    async def generate_response(self, system_prompt: str, user_prompt: str) -> Dict:
        response = await self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            response_format={"type": "json_object"}, # OpenAI hỗ trợ ép kiểu JSON
            temperature=0.1
        )
        return json.loads(response.choices[0].message.content)

# --- Gemini Provider ---
class GeminiProvider(LLMProvider):
    def __init__(self, api_key: str, model: str = "gemini-1.5-flash"):
        genai.configure(api_key=api_key)
        self.model_name = model
        self.model = genai.GenerativeModel(model)

    async def health_check(self):
        print(f"\n--- Checking Gemini ({self.model_name}) ---")
        try:
            loop = asyncio.get_event_loop()
            response = await loop.run_in_executor(None, self.model.generate_content, "ping")
            print("✅ Gemini API OK")
            print(f"Response: {response.text}")
            return True
        except Exception as e:
            print(f"❌ Gemini Error: {e}")
            return False

    async def generate_response(self, system_prompt: str, user_prompt: str) -> Dict:
        loop = asyncio.get_event_loop()
        combined_prompt = f"{system_prompt}\n\nUser Input: {user_prompt}"
        response = await loop.run_in_executor(None, self.model.generate_content, combined_prompt)
        return self._extract_json(response.text)

# --- Anthropic Provider ---
class AnthropicProvider(LLMProvider):
    def __init__(self, api_key: str, model: str = "claude-haiku-4-5-20251001"):
        self.client = AsyncAnthropic(api_key=api_key)
        self.model = model

    async def health_check(self):
        print(f"\n--- Checking Anthropic ({self.model}) ---")
        try:
            response = await self.client.messages.create(
                model=self.model,
                max_tokens=10,
                messages=[{"role": "user", "content": "ping"}]
            )
            print("✅ Anthropic API OK")
            print(f"Response: {response.content[0].text}")
            return True
        except Exception as e:
            print(f"❌ Anthropic Error: {e}")
            return False

    async def generate_response(self, system_prompt: str, user_prompt: str) -> Dict:
        response = await self.client.messages.create(
            model=self.model,
            max_tokens=4096,
            system=system_prompt,
            messages=[{"role": "user", "content": user_prompt}],
            temperature=0.1
        )
        return self._extract_json(response.content[0].text)

# ==========================================
# RUN HEALTH CHECK
# ==========================================
async def run_checks_1():
    # OpenAI
    if os.environ.get('OPENAI_API_KEY'):
        openai_llm = OpenAIProvider(api_key=os.environ['OPENAI_API_KEY'])
        await openai_llm.health_check()

    # Anthropic
    if os.environ.get('ANTHROPIC_API_KEY'):
        anthropic_llm = AnthropicProvider(api_key=os.environ['ANTHROPIC_API_KEY'])
        await anthropic_llm.health_check()

async def run_checks_2():
    system_p = "You are a helpful assistant. Always respond in valid JSON format."
    user_p = "Return a JSON object with a 'status': 'connected' and 'model': 'your_name'"

    print("\n" + "="*50)
    print("TESTING GENERATE_RESPONSE (JSON OUTPUT)")
    print("="*50)

    # --- OpenAI ---
    if os.environ.get('OPENAI_API_KEY'):
        try:
            openai_llm = OpenAIProvider(api_key=os.environ['OPENAI_API_KEY'], model="gpt-4o-mini")
            res = await openai_llm.generate_response(system_p, user_p)
            print(f"✅ OpenAI Result: {res} | Type: {type(res)}")
        except Exception as e:
            print(f"❌ OpenAI Gen Error: {e}")

    # --- Anthropic ---
    if os.environ.get('ANTHROPIC_API_KEY'):
        try:
            anthropic_llm = AnthropicProvider(api_key=os.environ['ANTHROPIC_API_KEY'])
            res = await anthropic_llm.generate_response(system_p, user_p)
            print(f"✅ Anthropic Result: {res} | Type: {type(res)}")
        except Exception as e:
            print(f"❌ Anthropic Gen Error: {e}")

# TEST

In [24]:
# --- Lớp mô phỏng để test ---
class MockProvider(LLMProvider):
    async def generate_response(self, system_prompt: str, user_prompt: str) -> Dict:
        pass
    async def health_check(self) -> bool:
        pass

def test_extract_json():
    provider = MockProvider()

    test_cases = [
        {
            "name": "JSON Object đơn thuần",
            "input": '{"status": "ok"}',
            "expected": {"status": "ok"}
        },
        {
            "name": "JSON Array đơn thuần",
            "input": '[{"id": 1}, {"id": 2}]',
            "expected": [{"id": 1}, {"id": 2}]
        },
        {
            "name": "Markdown Code Block (Object)",
            "input": "Dưới đây là kết quả:\n```json\n{\"key\": \"value\"}\n```\nHết.",
            "expected": {"key": "value"}
        },
        {
            "name": "Markdown Code Block (Array)",
            "input": "AI trả về danh sách:\n```json\n[1, 2, 3]\n```",
            "expected": [1, 2, 3]
        },
        {
            "name": "Văn bản lộn xộn (Array lồng Object)",
            "input": "Kết quả: [{\"data\": \"{text}\"}] - Hoàn tất",
            "expected": [{"data": "{text}"}]
        },
        {
            "name": "Object chứa Array bên trong",
            "input": '{"items": [1, 2, 3]}',
            "expected": {"items": [1, 2, 3]}
        },
        {
            "name": "Lỗi: Không có JSON",
            "input": "Đây chỉ là một câu nói bình thường.",
            "expected": None
        },
        {
            "name": "Lỗi: JSON sai cú pháp",
            "input": '{"key": "value"', # Thiếu dấu đóng }
            "expected": None
        },
        {
            "name": "Lỗi: Chuỗi rỗng",
            "input": "",
            "expected": None
        }
    ]

    print(f"{'STT':<5} | {'Tên Test Case':<35} | {'Kết quả':<10}")
    print("-" * 60)

    for i, case in enumerate(test_cases):
        result = provider._extract_json(case["input"])

        # Kiểm tra kết quả
        success = result == case["expected"]
        status = "✅ PASS" if success else "❌ FAIL"

        print(f"{i+1:<5} | {case['name']:<35} | {status}")
        if not success:
            print(f"   [!] Input: {case['input']}")
            print(f"   [!] Expected: {case['expected']}")
            print(f"   [!] Got: {result}")

if __name__ == "__main__":
    test_extract_json()

ERROR:root:Lỗi phân tách JSON: Không tìm thấy JSON. | Nội dung: Đây chỉ là một câu nói bình thường....
ERROR:root:Lỗi phân tách JSON: Expecting value: line 1 column 1 (char 0) | Nội dung: {"key": "value"...
ERROR:root:Lỗi phân tách JSON: Không tìm thấy JSON. | Nội dung: ...


STT   | Tên Test Case                       | Kết quả   
------------------------------------------------------------
1     | JSON Object đơn thuần               | ✅ PASS
2     | JSON Array đơn thuần                | ✅ PASS
3     | Markdown Code Block (Object)        | ✅ PASS
4     | Markdown Code Block (Array)         | ✅ PASS
5     | Văn bản lộn xộn (Array lồng Object) | ✅ PASS
6     | Object chứa Array bên trong         | ✅ PASS
7     | Lỗi: Không có JSON                  | ✅ PASS
8     | Lỗi: JSON sai cú pháp               | ✅ PASS
9     | Lỗi: Chuỗi rỗng                     | ✅ PASS


In [12]:
await run_checks_1()


--- Checking OpenAI (gpt-4o-mini) ---
✅ OpenAI API OK
Response: Pong! How can I assist you today?

--- Checking Anthropic (claude-haiku-4-5-20251001) ---
✅ Anthropic API OK
Response: pong 🏓

How can I


In [15]:
await run_checks_2()


TESTING GENERATE_RESPONSE (JSON OUTPUT)
✅ OpenAI Result: {'status': 'connected', 'model': 'your_name'} | Type: <class 'dict'>
✅ Anthropic Result: {'status': 'connected', 'model': 'Claude'} | Type: <class 'dict'>


# DATA EVALUATOR

In [65]:
# CHUẨN HÓA NHÃN: Xóa khoảng trắng và chuyển về chữ thường

class EvaluatorData:
    """Đánh giá chất lượng dữ liệu và sự thống nhất giữa các Agent"""

    @staticmethod
    def calculate_reliability(row: Dict) -> bool:
        """
        Kiểm tra bản ghi có đủ độ tin cậy để làm nhãn chuẩn (Gold Label) hay không.
        """
        try:
            t_label = (row.get('final_label', ''))
            v_label = (row.get('verifier_label', ''))

            print(t_label , v_label)

            # So khớp nhãn
            is_match = (t_label == v_label) and t_label != ""

            # Không có ảo giác
            no_hallucination = not row.get('hallucination_detected', True)

            # Điểm logic >= 4
            logic_ok = float(row.get('logic_consistency_score', 0)) >= 4.0

            # Độ tự tin trung bình >= 0.8
            t_conf = float(row.get('confidence_score', 0))
            v_conf = float(row.get('verifier_confidence', 0))
            confidence_ok = ((t_conf + v_conf) / 2) >= 0.8

            return is_match and no_hallucination and logic_ok and confidence_ok
        except:
            return False


    @staticmethod
    def get_statistics_metrics(df: pd.DataFrame) -> Dict[str, Any]:
        """Thống kê chi tiết chất lượng của toàn bộ dataset"""
        total = len(df)
        if total == 0: return {}

        # Thêm cột tin cậy nếu chưa có
        if 'is_reliable' not in df.columns:
            df['is_reliable'] = df.apply(EvaluatorData.calculate_reliability, axis=1)

        stats = {
            'total_samples': total,
            'reliable_samples': int(df['is_reliable'].sum()),
            'reliability_rate': float(df['is_reliable'].mean()),
            'avg_teacher_confidence': float(df['confidence_score'].mean()),
            'avg_verifier_confidence': float(df.get('verifier_confidence', 0).mean()),
            'hallucination_rate': float(df.get('hallucination_detected', 0).mean()),
             "avg_logic_score": float(df['logic_consistency_score'].fillna(0).mean()),
            # Tỷ lệ Teacher và Verifier cãi nhau
            'agreement_rate': ((df['final_label']) == (df['verifier_label'])).mean()
        }

        print("\n" + "-"*30)
        print("📊 THỐNG KÊ CHẤT LƯỢNG DATASET")
        print("-"*30)
        for k, v in stats.items():
            print(f"{k:25}: {v:.4f}" if isinstance(v, float) else f"{k:25}: {v}")
        print("-"*30)

        return stats

def test_evaluator_data():
    print("=== ĐANG CHẠY TEST CASE CHO EVALUATOR DATA ===\n")

    # 1. TẠO DỮ LIỆU MOCK (Dựa trên CSV của bạn nhưng thêm các cột Verifier)
    # Chúng ta sẽ tạo 4 trường hợp điển hình:
    mock_data = [
        {
            "input_text": "mng ơi mik mới mua cái đt mới xịn xò lắm lun",
            "final_label": "Constructive/Clean",
            "verifier_label": "Constructive/Clean", # Khớp nhãn
            "confidence_score": 0.98,
            "verifier_confidence": 0.95,           # Trung bình > 0.8
            "hallucination_detected": False,       # Không ảo giác
            "logic_consistency_score": 5,          # Logic tốt
        }, # => KẾT QUẢ MONG ĐỢI: True (Reliable)

        {
            "input_text": "clgt sao m lại làm thế vs t 😡",
            "final_label": "Explicit Hostility",
            "verifier_label": "Constructive/Clean", # SAI KHÁC NHÃN
            "confidence_score": 0.9,
            "verifier_confidence": 0.8,
            "hallucination_detected": False,
            "logic_consistency_score": 4,
        }, # => KẾT QUẢ MONG ĐỢI: False (Disagreement)

        {
            "input_text": "Hôm nay t đi học trễ vcl 😂😂😂",
            "final_label": "Explicit Hostility",
            "verifier_label": "Explicit Hostility",
            "confidence_score": 0.7,               # ĐỘ TỰ TIN THẤP (0.7+0.7)/2 = 0.7 < 0.8
            "verifier_confidence": 0.7,
            "hallucination_detected": False,
            "logic_consistency_score": 4,
        }, # => KẾT QUẢ MONG ĐỢI: False (Low confidence)

        {
            "input_text": "Giỏi quá vcl cả họ tự hào smirk",
            "final_label": "Implicit Toxicity",
            "verifier_label": "Implicit Toxicity",
            "confidence_score": 0.95,
            "verifier_confidence": 0.9,
            "hallucination_detected": True,        # CÓ ẢO GIÁC
            "logic_consistency_score": 5,
        }  # => KẾT QUẢ MONG ĐỢI: False (Hallucination)
    ]

    df_test = pd.DataFrame(mock_data)

    # 2. TEST HÀM 1: calculate_reliability
    print("--- Test: calculate_reliability ---")
    for i, row in enumerate(mock_data):
        is_reliable = EvaluatorData.calculate_reliability(row)
        status = "✅ PASS" if (i == 0 and is_reliable) or (i > 0 and not is_reliable) else "❌ FAIL"
        print(f"Sample {i+1}: Reliable={is_reliable} | {status}")

    # 3. TEST HÀM 2: get_statistics_metrics
    print("\n--- Test: get_statistics_metrics ---")
    stats = EvaluatorData.get_statistics_metrics(df_test)

    # Kiểm tra các chỉ số quan trọng
    assert stats['total_samples'] == 4
    assert stats['reliable_samples'] == 1
    assert stats['agreement_rate'] == 0.75 # 1/4 mẫu bị lệch nhãn
    print("\n=> Kiểm tra Assertions: Hoàn tất (Dữ liệu thống kê chính xác)")

if __name__ == "__main__":
    test_evaluator_data()

=== ĐANG CHẠY TEST CASE CHO EVALUATOR DATA ===

--- Test: calculate_reliability ---
Constructive/Clean Constructive/Clean
Sample 1: Reliable=True | ✅ PASS
Explicit Hostility Constructive/Clean
Sample 2: Reliable=False | ✅ PASS
Explicit Hostility Explicit Hostility
Sample 3: Reliable=False | ✅ PASS
Implicit Toxicity Implicit Toxicity
Sample 4: Reliable=False | ✅ PASS

--- Test: get_statistics_metrics ---
Constructive/Clean Constructive/Clean
Explicit Hostility Constructive/Clean
Explicit Hostility Explicit Hostility
Implicit Toxicity Implicit Toxicity

------------------------------
📊 THỐNG KÊ CHẤT LƯỢNG DATASET
------------------------------
total_samples            : 4
reliable_samples         : 1
reliability_rate         : 0.2500
avg_teacher_confidence   : 0.8825
avg_verifier_confidence  : 0.8375
hallucination_rate       : 0.2500
avg_logic_score          : 4.5000
agreement_rate           : 0.7500
------------------------------

=> Kiểm tra Assertions: Hoàn tất (Dữ liệu thống kê chính

In [21]:
import logging
# logging information
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("annotation.log", encoding='utf-8'),
        logging.StreamHandler()
    ]
)

# data verifier

In [68]:
class DataVerifier:
    def __init__(self, anthropic_provider: Any, system_prompt_path: str):
        self.provider = anthropic_provider
        self.system_prompt = self._load_prompt(system_prompt_path)
        self.evaluator = EvaluatorData()

    def get_statistics(self, df: pd.DataFrame):
        return self.evaluator.get_statistics_metrics(df)

    def _load_prompt(self, path: str) -> str:
        with open(path, 'r', encoding='utf-8') as f:
            return f.read().strip()

    def _format_batch_input(self, rows: List[Dict]) -> str:
        """Gom N mẫu vào 1 chuỗi theo format chuẩn định nghĩa trong Prompt"""
        count = len(rows)
        formatted_text = f"Hãy kiểm định chính xác {count} mẫu sau đây. Trả về một JSON ARRAY chứa đúng {count} đối tượng.\n\n"

        for i, row in enumerate(rows):
            # Gộp các bước phân tích nhỏ của Teacher thành Scaffolding
            scaffolding = (
                f"1. Semantic: {row.get('semantic_decoding', 'N/A')}; "
                f"2. Slang: {row.get('slang_interpretation', 'N/A')}; "
                f"3. Conflict: {row.get('contextual_conflict', 'N/A')}"
            )

            formatted_text += f"--- SAMPLE {i+1} ---\n"
            formatted_text += f"* **TEXT:** {row.get('input_text')}\n"
            formatted_text += f"* **EMOTION:** {row.get('input_emotion')}\n"
            formatted_text += f"* **TEACHER_reasoning_scaffolding:** {scaffolding}\n"
            formatted_text += f"* **TEACHER_thought_trace:** {row.get('thought_trace')}\n"
            formatted_text += f"* **TEACHER_LABEL:** {row.get('final_label')}\n\n"

        return formatted_text

    async def verify_chunk(self, chunk_rows: List[Dict]) -> List[Dict]:
        user_input = self._format_batch_input(chunk_rows)

        try:
            # Gọi Provider (Bây giờ đã dùng _extract_json mới)
            responses = await self.provider.generate_response(self.system_prompt, user_input)

            # Kiểm tra nếu responses là None hoặc không phải list
            if not isinstance(responses, list):
                responses = [responses] if responses else []

            verified_results = []
            for i, row in enumerate(chunk_rows):
                # Lấy kết quả từ AI, nếu AI trả thiếu thì gán giá trị mặc định
                v_res = responses[i] if i < len(responses) else {}

                verified_row = {
                    **row,
                    "verifier_label": v_res.get("verifier_label"),
                    "verifier_confidence": v_res.get("verifier_confidence", 0),
                    "hallucination_detected": v_res.get("hallucination_detected", False),
                    "logic_consistency_score": v_res.get("logic_consistency_score", 0),
                    "verifier_explanation": v_res.get("explanation", "No response from AI")
                }
                # Tính toán is_reliable dựa trên logic Evaluator
                verified_row["is_reliable"] = self.evaluator.calculate_reliability(verified_row)
                verified_results.append(verified_row)

            return verified_results

        except Exception as e:
            logging.error(f"Lỗi khi xử lý batch: {e}")
            return [{**r, "is_reliable": False, "error": str(e)} for r in chunk_rows]

    async def process_verification(self, input_csv: str, output_csv: str, samples_per_request: int = 5):
        df = pd.read_csv(input_csv)
        data = df.to_dict('records')
        all_results = []

        for i in range(0, len(data), samples_per_request):
            chunk = data[i : i + samples_per_request]
            logging.info(f"Đang xử lý mẫu {i} đến {i + len(chunk)}...")

            chunk_results = await self.verify_chunk(chunk)
            all_results.extend(chunk_results)

            if i + samples_per_request < len(data):
                await asyncio.sleep(5)

        verified_df = pd.DataFrame(all_results)
        verified_df.to_csv(output_csv, index=False, encoding='utf-8-sig')

        # --- IN STATISTICS ---
        print("\n" + "="*50)
        print("📊 TỔNG KẾT KIỂM ĐỊNH (PHASE 2)")
        print("="*50)
        self.evaluator.get_statistics_metrics(verified_df)

In [70]:
async def test_logic_phase_2():
    logging.info("--- ĐANG CHẠY TEST LOGIC PHASE 2 (VERIFIER) ---")

    # 1. Khởi tạo (Sử dụng model thực tế: claude-3-5-sonnet-latest)
    anthropic_llm = AnthropicProvider(
        api_key=os.environ['ANTHROPIC_API_KEY'],
        model="claude-sonnet-4-6"
    )
    # Đảm bảo đường dẫn prompt đúng
    verifier = DataVerifier(anthropic_provider=anthropic_llm, system_prompt_path='/content/prompt_version_2.txt')

    mock_teacher_results = [
        # 1. MẪU CHUẨN: Đồng thuận giữa Teacher và Verifier
        {
            "input_text": "mng ơi mik mới mua cái đt mới xịn xò lắm lun",
            "input_emotion": "happy",
            "semantic_decoding": "Người dùng mua điện thoại mới",
            "slang_interpretation": "xịn xò = đồ tốt, chất lượng cao",
            "contextual_conflict": "Không có mâu thuẫn giữa lời nói và cảm xúc vui vẻ",
            "thought_trace": "Người dùng chia sẻ niềm vui mua đồ mới, mang tính xây dựng.",
            "final_label": "Constructive/Clean",
            "confidence_score": 0.95
        },

        # 2. MẪU SAI (HALLUCINATION): Teacher không hiểu từ lóng thô tục
        {
            "input_text": "clgt sao m lại làm thế vs t 😡",
            "input_emotion": "angry",
            "semantic_decoding": "Câu hỏi về hành động của người khác",
            "slang_interpretation": "clgt = viết tắt thông thường", # SAI: Teacher không giải mã được độ tục
            "contextual_conflict": "Cảm xúc giận dữ phù hợp với câu hỏi",
            "thought_trace": "Đây là lời hỏi thăm trong lúc nóng giận nhưng không thù địch.", # SAI: Logic yếu
            "final_label": "Constructive/Clean", # SAI: Đáng lẽ phải là Explicit Hostility
            "confidence_score": 0.88
        },

        # 3. MẪU SAI (SARCASM): Teacher bị đánh lừa bởi câu chữ "khen"
        {
            "input_text": "Giỏi quá vcl cả họ tự hào",
            "input_emotion": "smirk",
            "semantic_decoding": "Khen ngợi sự giỏi giang và niềm tự hào gia đình",
            "slang_interpretation": "vcl = vãi cả lồn (trạng từ nhấn mạnh)",
            "contextual_conflict": "Lời khen nhưng đi kèm biểu cảm cười nhếch mép (smirk)",
            "thought_trace": "Người nói đang khen ngợi đối phương một cách nhiệt tình.", # SAI: Bỏ qua tín hiệu mỉa mai
            "final_label": "Constructive/Clean", # SAI: Đáng lẽ phải là Implicit Toxicity
            "confidence_score": 0.82
        }

        # # 4. MẪU MỚI: TẤN CÔNG BẢN DẠNG (Identity-Based Hate) - Teacher đánh nhãn quá nhẹ
        # {
        #     "input_text": "mấy thằng nhà quê lên tỉnh phá hoại quá",
        #     "input_emotion": "angry",
        #     "semantic_decoding": "Phàn nàn về người từ nông thôn lên thành phố",
        #     "slang_interpretation": "nhà quê = người ở nông thôn",
        #     "contextual_conflict": "Sự tức giận nhắm vào một nhóm người dựa trên nguồn gốc",
        #     "thought_trace": "Người dùng phàn nàn về hành vi phá hoại của người khác.",
        #     "final_label": "Explicit Hostility", # SAI: Nên là Identity-Based Hate (Phân biệt vùng miền)
        #     "confidence_score": 0.9
        # },

        # # 5. MẪU MỚI: TỪ TỤC NGHĨA TÍCH CỰC (Positive Profanity) - Teacher quá khắt khe
        # {
        #     "input_text": "vcl phim này đỉnh thực sự mng ơi 🎥🔥",
        #     "input_emotion": "happy",
        #     "semantic_decoding": "Khen ngợi bộ phim rất hay",
        #     "slang_interpretation": "vcl = từ cảm thán thô tục",
        #     "contextual_conflict": "Dùng từ tục nhưng cảm xúc lại là hào hứng và khen ngợi",
        #     "thought_trace": "Câu nói chứa từ ngữ thô tục 'vcl' nên vi phạm tiêu chuẩn cộng đồng.",
        #     "final_label": "Explicit Hostility", # SAI: Nên là Constructive/Clean (Dùng từ tục để nhấn mạnh sự yêu thích)
        #     "confidence_score": 0.92
        # }
    ]


    # 3. Thực thi
    results = await verifier.verify_chunk(mock_teacher_results)

    df_res = pd.DataFrame(results)

    print("\n" + "="*30 + " CHI TIẾT KẾT QUẢ TEST " + "="*30)
    for res in results:
        print(f"\n[-] Text: {res['input_text']}")
        print(f"[-] Teacher: {res['final_label']} | Verifier: {res['verifier_label']}")
        print(f"[>] Logic Score: {res['logic_consistency_score']}/5 | Hallucination: {res['hallucination_detected']}")
        print(f"[!] Reliable: {'✅ YES' if res['is_reliable'] else '❌ NO'}")
        print(f"[*] Note: {res['verifier_explanation']}")
        print("-" * 60)

    # In thống kê
    verifier.get_statistics(df_res)
    return df_res

# Chạy test (trong môi trường hỗ trợ await như Colab/Jupyter)
# await test_logic_phase_2()

In [71]:
res = await test_logic_phase_2()

Constructive/Clean Constructive/Clean
Constructive/Clean Implicit Toxicity
Constructive/Clean Implicit Toxicity

============================== CHI TIẾT KẾT QUẢ TEST ==============================

[-] Text: mng ơi mik mới mua cái đt mới xịn xò lắm lun
[-] Teacher: Constructive/Clean | Verifier: Constructive/Clean
[>] Logic Score: 5/5 | Hallucination: False
[!] Reliable: ✅ YES
[*] Note: Văn bản hoàn toàn tích cực: người dùng chia sẻ niềm vui mua điện thoại mới với cộng đồng ('mng ơi'). Slang 'xịn xò' được giải mã đúng (= chất lượng cao, tốt). Cảm xúc 'happy' nhất quán với nội dung. Không có dấu hiệu độc hại hay thù địch. Teacher AI phân tích chính xác, lập luận mạch lạc.
------------------------------------------------------------

[-] Text: clgt sao m lại làm thế vs t 😡
[-] Teacher: Constructive/Clean | Verifier: Implicit Toxicity
[>] Logic Score: 2/5 | Hallucination: True
[!] Reliable: ❌ NO
[*] Note: Teacher AI mắc lỗi nghiêm trọng khi giải mã 'clgt': đây là viết tắt của 'cái lồn gì 

In [73]:
res

,input_text,input_emotion,semantic_decoding,slang_interpretation,contextual_conflict,thought_trace,final_label,confidence_score,verifier_label,verifier_confidence,hallucination_detected,logic_consistency_score,verifier_explanation,is_reliable
0,mng ơi mik mới mua cái đt mới xịn xò lắm lun,happy,Người dùng mua điện thoại mới,"xịn xò = đồ tốt, chất lượng cao",Không có mâu thuẫn giữa lời nói và cảm xúc vui vẻ,"Người dùng chia sẻ niềm vui mua đồ mới, mang t...",Constructive/Clean,0.95,Constructive/Clean,0.97,False,5,Văn bản hoàn toàn tích cực: người dùng chia sẻ...,True
1,clgt sao m lại làm thế vs t 😡,angry,Câu hỏi về hành động của người khác,clgt = viết tắt thông thường,Cảm xúc giận dữ phù hợp với câu hỏi,Đây là lời hỏi thăm trong lúc nóng giận nhưng ...,Constructive/Clean,0.88,Implicit Toxicity,0.82,True,2,Teacher AI mắc lỗi nghiêm trọng khi giải mã 'c...,False
2,Giỏi quá vcl cả họ tự hào,smirk,Khen ngợi sự giỏi giang và niềm tự hào gia đình,vcl = vãi cả lồn (trạng từ nhấn mạnh),Lời khen nhưng đi kèm biểu cảm cười nhếch mép ...,Người nói đang khen ngợi đối phương một cách n...,Constructive/Clean,0.82,Implicit Toxicity,0.88,True,2,Teacher AI bỏ qua tín hiệu quan trọng nhất: cả...,False
